In [65]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
sys.path.append('./scripts')  
import preprocesamiento
import feature_engineering
import model_lgb
import model_lgb_keepsimple
importlib.reload(preprocesamiento)
importlib.reload(model_lgb)
importlib.reload(model_lgb_keepsimple)
importlib.reload(feature_engineering)
warnings.filterwarnings("ignore")

# Experimento 7: 
- LGBM
- Usando funcion entrenamiento: model_lgb.semillerio_en_prediccion(train, test, version="v1")
- optuna = sqlite:///optuna_studies_v14.db
- Kaggle =  


##### Levantamos el dataset con target ya calculado

In [37]:
df = pd.read_csv("./datasets/periodo_x_producto_con_target.csv", sep=',', encoding='utf-8')
df.shape

(31362, 19)

In [25]:
df[df['target'].isna()][['product_id', 'periodo', 'tn', 'target']]
# 20034 , 201905

,product_id,periodo,tn,target
34,20001,201911,1397.37231,NaN
35,20001,201912,1504.68856,NaN
70,20002,201911,1423.57739,NaN
71,20002,201912,1087.30855,NaN
106,20003,201911,948.29393,NaN
...,...,...,...,...
31344,21274,201708,0.00867,NaN
31353,21276,201911,0.03341,NaN
31354,21276,201912,0.00892,NaN
31360,21281,201707,0.00000,NaN


In [26]:
columnas_baseline = df.columns.tolist()
columnas_baseline

['product_id',
 'periodo',
 'nacimiento_producto',
 'muerte_producto',
 'mes_n',
 'total_meses',
 'producto_nuevo',
 'ciclo_de_vida_inicial',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'sku_size',
 'stock_final',
 'tn',
 'plan_precios_cuidados',
 'cust_request_qty',
 'cust_request_tn',
 'target']

##### Preprocesamiento a la minima expresión :)

In [38]:
df = feature_engineering.create_category_features_cat1(df)
df = feature_engineering.create_category_features_cat2(df)
df = feature_engineering.create_category_features_cat3(df)

##### aplicamos OHE
# df = preprocesamiento.aplicarOHE(df)
df.shape

(31362, 28)

### Feature Engineering

##### Neural Prophet

In [40]:
neural_prophet_fe = pd.read_csv("./datasets/features_neuralprophet_completo.csv", sep=',', encoding='utf-8')
neural_prophet_fe['ds'] = pd.to_datetime(neural_prophet_fe['ds'], errors='coerce')
# Versión alternativa más robusta:
neural_prophet_fe['periodo'] = neural_prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
neural_prophet_fe = neural_prophet_fe[['periodo', 'product_id', 'trend', "season_yearly", "season_monthly"]]
df = df.merge(neural_prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 31)

##### Prophet

In [41]:
prophet_fe = pd.read_csv("./datasets/prophet_features_tn_zscore.csv", sep=',', encoding='utf-8')
prophet_fe['ds'] = pd.to_datetime(prophet_fe['ds'], errors='coerce')
prophet_fe['periodo'] = prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
prophet_fe = prophet_fe[['periodo', 'product_id', 'trend_add', "yearly_add", "additive_terms", 'trend_mult', 'yearly_mult', 'multiplicative_terms']]
df = df.merge(prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 37)

##### FE Moviles

In [43]:
df = feature_engineering.get_lags(df, "tn", 201912)
df = feature_engineering.get_delta_lags(df, "tn", 24)
df = feature_engineering.get_rolling_means(df, "tn", 201912)
df = feature_engineering.get_rolling_stds(df, "tn", 201912)
df = feature_engineering.get_rolling_mins(df, "tn", 201912)
df = feature_engineering.get_rolling_maxs(df, "tn", 201912)
df = feature_engineering.get_rolling_medians(df, "tn", 201912)
df = feature_engineering.get_rolling_skewness(df, "tn", 201912)
df = feature_engineering.get_autocorrelaciones(df, "tn", 201912)
df.shape

(31362, 599)

In [8]:
df = feature_engineering.get_lags(df, "cust_request_qty", 201912)
df = feature_engineering.get_delta_lags(df, "cust_request_qty", 24)
df = feature_engineering.get_rolling_means(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_stds(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_mins(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_maxs(df, "cust_request_qty", 201912)

In [ ]:
df = feature_engineering.get_lags(df, "cust_request_qty", 201912)
df = feature_engineering.get_delta_lags(df, "cust_request_qty", 24)
df = feature_engineering.get_rolling_means(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_stds(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_mins(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_maxs(df, "cust_request_qty", 201912)
df.shape

(31362, 1500)

##### FE Diana

In [45]:
df = feature_engineering.calcular_diferencia_con_medias_moviles(df)
df = feature_engineering.calcular_ratios_con_medias_moviles(df)
df.shape

(31362, 635)

##### FE Moviles sobre otras variables

In [56]:
# #  stock final
# df = feature_engineering.get_lagsEspecificos(df, col='stock_final_zscore')
# df = feature_engineering.get_delta_lags_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_means_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_stds_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_medians_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_mins_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='stock_final_zscore')

#  cust_request_qty
# df = feature_engineering.get_lagsEspecificos(df, col='cust_request_qty')
# df = feature_engineering.get_delta_lags_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_means_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_stds_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_mins_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_medians_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='cust_request_qty')

##### FE Calendario

In [47]:
df = feature_engineering.generar_ids(df)
df = feature_engineering.get_componentesTemporales(df)
df = feature_engineering.get_anomaliasPoliticas(df)
# df = feature_engineering.descomposicion_serie_temporal(df, col='tn')
df.shape

(31362, 660)

##### FE sobre FE

In [33]:
df = feature_engineering.chatGPT_features_serie(df, "tn")
df = feature_engineering.mes_con_feriado(df)
df.shape

(31362, 689)

##### Variables Exogenas

In [49]:
df = feature_engineering.get_dolar(df)
df = feature_engineering.get_IPC(df)
df['ipc'] = df['ipc'].str.replace(',', '.').astype(float)
df['dolar'] = df['dolar'].str.replace(',', '.').astype(float)
# df.drop(columns=['ds'], inplace=True)
#df.fillna(0, inplace=True) ##### EXPERIMENTAR
df = feature_engineering.correlacion_exogenas(df)
df = feature_engineering.get_mes_receso_escolar(df)
df.shape

(31362, 666)

##### Nuevas FE

In [52]:
df = feature_engineering.create_ratio_features(df)
df = feature_engineering.enhance_lifecycle_features(df)
# df = feature_engineering.create_category_features(df)
df = feature_engineering.create_regime_features(df)
df = feature_engineering.create_nonlinear_trends(df)
df = feature_engineering.create_temporal_interactions(df)
df = feature_engineering.create_asymmetric_window_features(df)
# df = feature_engineering.recomendaciones_deepseek(df)
df = feature_engineering.get_nuevas_features(df)
df.shape

(31362, 702)

##### Ceros

In [ ]:
df = feature_engineering.agregar_ceros_consecutivos_atras(df, col_tn='tn_original')
df = feature_engineering.agregar_no_ceros_consecutivos_atras(df, col_tn='tn_original')
df = feature_engineering.agregar_ceros_ultimos_n_meses(df, ventanas=[1,2,3,4,5,6,12], col_tn='tn_original')
df = feature_engineering.agregar_min_max_ult_n(df, n_list=(1,2,3,4,5,6,12), col_tn='tn_original')
df.shape

##### Elimino aquellas que no sirven

In [13]:
importantes = pd.read_csv("./feature_importance/exp04_3.csv", sep=',', encoding='utf-8')
no_importantes = importantes[importantes['importance'] <= 100]
no_importantes = no_importantes[~no_importantes['feature'].isin(columnas_baseline)]
no_importantes

,feature,importance
67,tn_vs_prev_year,100
68,tn_delta_lag5_lag8,99
69,tn_rolling_min_4,96
70,tn_lag_5,95
71,tn_rolling_std_6,95
...,...,...
677,dia_del_year,0
678,cat2_TE,0
680,cat2_PIEL1,0
681,cat2_OTROS,0


In [14]:
cols_a_eliminar = no_importantes.feature.unique()
print(f"Antes de eliminar: {df.shape[1]} columnas")
df = df.drop(columns=cols_a_eliminar, errors='ignore')
print(f"Después de eliminar: {df.shape[1]} columnas")

Antes de eliminar: 1144 columnas
Después de eliminar: 684 columnas


Eliminar object/categorical columnas

In [15]:
df = df.select_dtypes(exclude=['datetime', 'datetime64', 'object'])

Train Test Split

In [54]:
training = [
    201701, 201702, 201703, 201704, 201705, 201706, 201707, 201708, 201709,
    201710, 201711, 201712, 201801, 201802, 201803, 201804, 201805,
    201806, 201807, 201808, 201809, 201810, 201811, 201812,
    201901, 201902, 201903, 201904, 201905, 201906, 201907, 201908
]

validation = [
    201909
]


testing = [
    201910
]

prediction = [
    201912  
]

In [55]:
df_train = df[df['periodo'].isin(training)].copy()
df_val = df[df['periodo'].isin(validation)].copy()
df_test = df[df['periodo'].isin(testing)].copy()
df_pred = df[df['periodo'].isin(prediction)].copy()

gc.collect()

327

Entrenamiento

In [59]:
# Hay casos como este donde el producto muere en 2019006, por lo tanto tienen los dos ultimos target vacios.
# df_train[df_train['product_id']==20034][['product_id', 'periodo', 'tn', 'target']]
# 20034 , 201905

In [60]:
df_train['target'].fillna(0, inplace=True)
df_val['target'].fillna(0, inplace=True)
df_test['target'].fillna(0, inplace=True)

In [61]:
model_lgb_keepsimple.optimizar_con_optuna_sin5FCV_con_semillerio_db(df_train=df_train, df_val=df_val, version="v24")


Para visualizar los resultados en tiempo real:
1. Abre otra terminal y ejecuta:
   optuna-dashboard sqlite:///optuna_studies_v24.db
2. Abre en tu navegador: http://127.0.0.1:8080/


[I 2025-07-15 20:59:12,223] Using an existing study with name 'lightgbm_optimization_v24' instead of creating a new one.
[I 2025-07-15 20:59:43,700] Trial 2 finished with value: 0.24713381488498531 and parameters: {'num_leaves': 47, 'learning_rate': 0.2536999076681772, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8795975452591109, 'bagging_freq': 2, 'lambda_l1': 2.5348407664333426e-07, 'lambda_l2': 3.3323645788192616e-08, 'min_child_samples': 45, 'max_depth': 7, 'max_bin': 383, 'min_data_in_leaf': 21, 'extra_trees': True, 'early_stopping_rounds': 18, 'path_smooth': 0.18182496720710062, 'min_gain_to_split': 0.09170225492671691}. Best is trial 2 with value: 0.24713381488498531.


Mejor trial hasta ahora: TFE=0.247134, Parámetros={'num_leaves': 47, 'learning_rate': 0.2536999076681772, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8795975452591109, 'bagging_freq': 2, 'lambda_l1': 2.5348407664333426e-07, 'lambda_l2': 3.3323645788192616e-08, 'min_child_samples': 45, 'max_depth': 7, 'max_bin': 383, 'min_data_in_leaf': 21, 'extra_trees': True, 'early_stopping_rounds': 18, 'path_smooth': 0.18182496720710062, 'min_gain_to_split': 0.09170225492671691}


[I 2025-07-15 21:00:56,806] Trial 3 finished with value: 0.27233624554335006 and parameters: {'num_leaves': 41, 'learning_rate': 0.05958389350068958, 'feature_fraction': 0.7727780074568463, 'bagging_fraction': 0.7873687420594125, 'bagging_freq': 7, 'lambda_l1': 1.8007140198129195e-07, 'lambda_l2': 4.258943089524393e-06, 'min_child_samples': 25, 'max_depth': 6, 'max_bin': 414, 'min_data_in_leaf': 36, 'extra_trees': False, 'early_stopping_rounds': 11, 'path_smooth': 0.6075448519014384, 'min_gain_to_split': 0.08526206184364576}. Best is trial 2 with value: 0.24713381488498531.


Mejor trial hasta ahora: TFE=0.247134, Parámetros={'num_leaves': 47, 'learning_rate': 0.2536999076681772, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8795975452591109, 'bagging_freq': 2, 'lambda_l1': 2.5348407664333426e-07, 'lambda_l2': 3.3323645788192616e-08, 'min_child_samples': 45, 'max_depth': 7, 'max_bin': 383, 'min_data_in_leaf': 21, 'extra_trees': True, 'early_stopping_rounds': 18, 'path_smooth': 0.18182496720710062, 'min_gain_to_split': 0.09170225492671691}


...

[I 2025-07-17 03:49:47,676] Trial 500 finished with value: 0.2623521055976997 and parameters: {'num_leaves': 95, 'learning_rate': 0.011736645915993285, 'feature_fraction': 0.8933902586971586, 'bagging_fraction': 0.9418487145981, 'bagging_freq': 9, 'lambda_l1': 1.5146949269264286e-06, 'lambda_l2': 3.324055340104984e-08, 'min_child_samples': 33, 'max_depth': 8, 'max_bin': 225, 'min_data_in_leaf': 40, 'extra_trees': False, 'early_stopping_rounds': 49, 'path_smooth': 0.579297289099909, 'min_gain_to_split': 0.31470465847822776}. Best is trial 415 with value: 0.21891901764809552.


Mejor trial hasta ahora: TFE=0.218919, Parámetros={'num_leaves': 94, 'learning_rate': 0.011828769582017987, 'feature_fraction': 0.8730198358476882, 'bagging_fraction': 0.959933825640181, 'bagging_freq': 8, 'lambda_l1': 8.833494682319988e-08, 'lambda_l2': 7.062421979561123e-05, 'min_child_samples': 32, 'max_depth': 8, 'max_bin': 242, 'min_data_in_leaf': 34, 'extra_trees': True, 'early_stopping_rounds': 48, 'path_smooth': 0.39251848468006784, 'min_gain_to_split': 0.3363925701702316}


[I 2025-07-17 03:52:48,131] Trial 501 finished with value: 0.23731584208495055 and parameters: {'num_leaves': 99, 'learning_rate': 0.0105802080224289, 'feature_fraction': 0.9085462386027983, 'bagging_fraction': 0.9307935360875796, 'bagging_freq': 4, 'lambda_l1': 0.3519615272484014, 'lambda_l2': 8.650957654167965e-05, 'min_child_samples': 29, 'max_depth': 8, 'max_bin': 122, 'min_data_in_leaf': 81, 'extra_trees': True, 'early_stopping_rounds': 50, 'path_smooth': 0.6329891840655055, 'min_gain_to_split': 0.36172206631018705}. Best is trial 415 with value: 0.21891901764809552.


Mejor trial hasta ahora: TFE=0.218919, Parámetros={'num_leaves': 94, 'learning_rate': 0.011828769582017987, 'feature_fraction': 0.8730198358476882, 'bagging_fraction': 0.959933825640181, 'bagging_freq': 8, 'lambda_l1': 8.833494682319988e-08, 'lambda_l2': 7.062421979561123e-05, 'min_child_samples': 32, 'max_depth': 8, 'max_bin': 242, 'min_data_in_leaf': 34, 'extra_trees': True, 'early_stopping_rounds': 48, 'path_smooth': 0.39251848468006784, 'min_gain_to_split': 0.3363925701702316}
Estudio guardado en: sqlite:///optuna_studies_v24.db

Mejores hiperparámetros encontrados:
num_leaves: 94
learning_rate: 0.011828769582017987
feature_fraction: 0.8730198358476882
bagging_fraction: 0.959933825640181
bagging_freq: 8
lambda_l1: 8.833494682319988e-08
lambda_l2: 7.062421979561123e-05
min_child_samples: 32
max_depth: 8
max_bin: 242
min_data_in_leaf: 34
extra_trees: True
early_stopping_rounds: 48
path_smooth: 0.39251848468006784
min_gain_to_split: 0.3363925701702316


(<optuna.study.study.Study at 0x21780e0d510>,
 {'num_leaves': 94,
  'learning_rate': 0.011828769582017987,
  'feature_fraction': 0.8730198358476882,
  'bagging_fraction': 0.959933825640181,
  'bagging_freq': 8,
  'lambda_l1': 8.833494682319988e-08,
  'lambda_l2': 7.062421979561123e-05,
  'min_child_samples': 32,
  'max_depth': 8,
  'max_bin': 242,
  'min_data_in_leaf': 34,
  'extra_trees': True,
  'early_stopping_rounds': 48,
  'path_smooth': 0.39251848468006784,
  'min_gain_to_split': 0.3363925701702316,
  'objective': 'regression',
  'metric': 'rmse',
  'boosting_type': 'gbdt',
  'verbosity': -1})

Prediccion Test

In [66]:
df_prediccion_test = model_lgb_keepsimple.semillerio_en_prediccion_con_pesos(df_train, df_val, df_test, version="v24")

In [67]:
df_prediccion_test

,periodo,product_id,target,pred
28656,201910,20001,1504.68856,1334.000601
28657,201910,20002,1087.30855,1376.701029
28658,201910,20003,892.50129,837.634090
28659,201910,20004,637.90002,568.996552
28660,201910,20005,593.24443,596.094511
...,...,...,...,...
29567,201910,21266,0.05121,0.212243
29568,201910,21267,0.01569,0.110493
29569,201910,21269,0.00000,-0.609200
29570,201910,21271,0.00298,-0.326782


Productos

In [69]:
productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")

In [71]:
df_promedios = model_lgb.promedio_12_meses_780p()
df_promedios

,product_id,tn
0,20001,1454.732720
1,20002,1175.437142
2,20003,784.976407
3,20004,627.215328
4,20005,668.270104
...,...,...
775,21263,0.029993
776,21265,0.089541
777,21266,0.094659
778,21267,0.092835


Filtramos

In [72]:
df_prediccion_test = df_prediccion_test[df_prediccion_test['product_id'].isin(productos_ok['product_id'].unique())]

In [73]:
df_prediccion_test = df_prediccion_test.merge(df_promedios, on=['product_id'], how='left', suffixes=('', '_promedio'))
df_prediccion_test

,periodo,product_id,target,pred,tn
0,201910,20001,1504.68856,1334.000601,1454.732720
1,201910,20002,1087.30855,1376.701029,1175.437142
2,201910,20003,892.50129,837.634090,784.976407
3,201910,20004,637.90002,568.996552,627.215328
4,201910,20005,593.24443,596.094511,668.270104
...,...,...,...,...,...
775,201910,21263,0.01270,0.247541,0.029993
776,201910,21265,0.05007,0.456357,0.089541
777,201910,21266,0.05121,0.212243,0.094659
778,201910,21267,0.01569,0.110493,0.092835


Reemplazamos negativos por el promedio

In [75]:
df_prediccion_test.loc[df_prediccion_test["pred"] < 0, "pred"] = df_prediccion_test["tn"]

TFE

In [77]:
df_prediccion_test['tfe'] = (np.sum(np.abs(df_prediccion_test['target'] - df_prediccion_test['pred']))) / np.sum(df_prediccion_test['target'])
df_prediccion_test['tfe']

0      0.239665
1      0.239665
2      0.239665
3      0.239665
4      0.239665
         ...   
775    0.239665
776    0.239665
777    0.239665
778    0.239665
779    0.239665
Name: tfe, Length: 780, dtype: float64

Predicción kaggle

In [ ]:
df_prediccion_test = model_lgb_keepsimple.semillerio_en_prediccion_con_pesos(df_train, df_val, df_pred, version="v24")

In [80]:
df_prediccion_kaggle = df_prediccion_test.copy()
df_prediccion_kaggle = df_prediccion_kaggle[df_prediccion_kaggle['product_id'].isin(productos_ok['product_id'].unique())]
df_prediccion_kaggle

,periodo,product_id,target,pred
30476,201912,20001,NaN,1365.027653
30477,201912,20002,NaN,1114.254525
30478,201912,20003,NaN,656.971716
30479,201912,20004,NaN,487.696267
30480,201912,20005,NaN,566.637911
...,...,...,...,...
31355,201912,21263,NaN,-0.987871
31357,201912,21265,NaN,-0.207048
31358,201912,21266,NaN,-0.991461
31359,201912,21267,NaN,-1.202688


In [81]:
df_prediccion_kaggle = df_prediccion_kaggle.merge(df_promedios, on=['product_id'], how='left', suffixes=('', '_promedio'))
df_prediccion_kaggle

,periodo,product_id,target,pred,tn
0,201912,20001,NaN,1365.027653,1454.732720
1,201912,20002,NaN,1114.254525,1175.437142
2,201912,20003,NaN,656.971716,784.976407
3,201912,20004,NaN,487.696267,627.215328
4,201912,20005,NaN,566.637911,668.270104
...,...,...,...,...,...
775,201912,21263,NaN,-0.987871,0.029993
776,201912,21265,NaN,-0.207048,0.089541
777,201912,21266,NaN,-0.991461,0.094659
778,201912,21267,NaN,-1.202688,0.092835


In [82]:
df_prediccion_kaggle.loc[df_prediccion_kaggle["pred"] < 0, "pred"] = df_prediccion_kaggle["tn"]

In [83]:
df_prediccion_kaggle = df_prediccion_kaggle[['product_id', 'pred']]
df_prediccion_kaggle.rename(columns={'pred': 'tn'}, inplace=True)
df_prediccion_kaggle.to_csv("./datasets/prediccion_exp09_lgb_v3.csv", index=False, sep=",", encoding='utf-8')